<a href="https://colab.research.google.com/github/rickycarrillo3/Deepfake_Detection_Project/blob/main/score_correlation_publish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Correlation Across Models

Reads every prediction CSV in a Drive folder and compiles **multiple tables** -- one for each detected generation model group -- each indexed by `filepath` (sorted), with two columns per model:

- `score_<model>`
- `delta_vector_<model>` — 1 if `score >= 0.5`, else 0

The generation model group name is extracted from the filename (the part before the first underscore). The individual model name is whatever follows this group prefix and precedes `predictions` in the filename (e.g. `stylegan2_freqnet_GAN_predictions.csv` -> `freqnet_GAN` for the `stylegan2` group). Adding a new model just means dropping its CSV into the folder and re-running.

For each of these tables, this computes the score correlation across models (Pearson for the continuous score vectors) and delta correlation (Pearson + Spearman for the binary delta vectors), and renders each as an annotated heatmap. It also calculates oracle accuracy for each group, taking into account that the correct label for `ffhq` is 0 and for all other subsets is 1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import filecmp
from sklearn.metrics import roc_auc_score

## Helper functions
FNR, metrics, and calibrated score functions are here


In [ ]:
def compute_fnr(combined, dataset_name, delta_threshold=0.5):
    # Fake subsets are those with label 1 (anything but ffhq).
    # FNR is the rate of false negatives (predicting 0 when it's 1).
    if 'ffhq' in dataset_name.lower():
        return None

    delta_cols = [c for c in combined.columns if c.startswith('delta_vector_')]
    fnr_values = {}
    for col in delta_cols:
        model_name = col.replace('delta_vector_', '')
        fnr = (combined[col] == 0).sum() / len(combined)
        fnr_values[model_name] = fnr

    return fnr_values


In [ ]:
def get_calibrated_score(score, sorted_ffhq_scores):
    # The calibrated score is the count of FFHQ scores >= the given score
    # Use searchsorted to quickly find the count in a sorted array
    # sorted_ffhq_scores must be sorted ascendingly
    idx = np.searchsorted(sorted_ffhq_scores, score, side='left')
    return len(sorted_ffhq_scores) - idx


In [ ]:
def compute_oracle_accuracy(combined, dataset_name):
    if combined is None:
        print(f"[{dataset_name}] No combined data available for oracle accuracy computation.")
        return None

    delta_cols = [c for c in combined.columns if c.startswith('delta_vector_')]

    # Exclude columns that are all 1s (100% positive rate)
    valid_delta_cols = [c for c in delta_cols if combined[c].mean() < 1.0]

    excluded_cols = set(delta_cols) - set(valid_delta_cols)
    if excluded_cols:
        excluded_names = [c.replace('delta_vector_', '') for c in excluded_cols]
        print(f"[{dataset_name}] Excluded models with 100% positive rate: {', '.join(excluded_names)}")

    if not valid_delta_cols:
        print(f"[{dataset_name}] No valid models left to compute oracle accuracy.")
        return None

    # The oracle predicts 1 (fake) if ANY model predicts 1 (hard-OR)
    oracle_prediction = combined[valid_delta_cols].any(axis=1).astype(int)

    if 'ffhq' in dataset_name.lower():
        # For ffhq (label 0), oracle is correct if it predicts 0
        correct_entries = (oracle_prediction == 0).sum()
    else:
        # For other datasets (label 1), oracle is correct if it predicts 1
        correct_entries = (oracle_prediction == 1).sum()

    # Calculate accuracy: sum of entries / total number of entries
    total_entries = len(oracle_prediction)
    oracle_accuracy = correct_entries / total_entries

    print(f"[{dataset_name}] Oracle Accuracy: {oracle_accuracy:.2%} ({correct_entries} / {total_entries})")
    print("-" * 50)

    return oracle_accuracy

# The loop for this function is now integrated into cell crurM6bdv4VE.

In [ ]:
def extract_model_name(path, group_prefix):
    """'stylegan2_freqnet_GAN_predictions.csv' with group_prefix='stylegan2_' -> 'freqnet_GAN'"""
    fname = os.path.basename(path)
    stem = fname[:-4] if fname.lower().endswith('.csv') else fname

    # Strip the provided group_prefix, performing a case-insensitive check
    if stem.lower().startswith(group_prefix.lower()):
        stem = stem[len(group_prefix):]
    else:
        # Fallback if prefix doesn't match for some reason, or if it's an "ungrouped" file
        # In the context of this refactor, this branch should ideally not be hit
        # if the grouping logic in crurM6bdv4VE is sound.
        pass

    return re.sub(r'_?predictions$', '', stem, flags=re.IGNORECASE)

In [ ]:
def build_combined_table(csv_paths, group_prefix, delta_threshold=0.5):
    """Compile a group of prediction CSVs into one table indexed by filepath,
    with score_<model> / delta_vector_<model> columns.
    Files that resolve to a duplicate model name are skipped with a warning."""
    if not csv_paths:
        return None

    frames = []
    seen_models = {}
    for path in csv_paths:
        try:
            df = pd.read_csv(path)
        except Exception as e:
            print(f"ERROR: Could not read CSV {os.path.basename(path)}: {e}. Skipping.")
            continue

        if 'filepath' not in df.columns:
            print(f"WARNING: CSV {os.path.basename(path)} is missing the 'filepath' column. Skipping this file.")
            continue

        model = extract_model_name(path, group_prefix) # Pass group_prefix here
        if model in seen_models:
            print(f'WARNING: model name "{model}" is not unique -- skipping {os.path.basename(path)} '
                  f'(already used by {seen_models[model]}). Rename the file so the model name is unique.')
            continue
        seen_models[model] = os.path.basename(path)

        df = df.set_index('filepath').sort_index()

        sub = pd.DataFrame(index=df.index)
        sub[f'score_{model}'] = df['score']
        sub[f'delta_vector_{model}'] = (df['score'] >= delta_threshold).astype(int)
        # Removed the creation of error_<model> column as per user request
        frames.append(sub)

    if not frames:
        return None

    return pd.concat(frames, axis=1, join='inner').sort_index()

In [ ]:
def compute_correlations(combined):
    """Pearson correlation of score_* columns; Pearson + Spearman of delta_vector_* columns.

    NOTE: delta_vector_* columns that are constant (the model scored >= 0.5 on
    every single image, or on none of them -- i.e. a 100% or 0% positive rate)
    are excluded from the delta correlation heatmaps. A constant column has
    zero variance, so its correlation with every other model is mathematically
    undefined (NaN); leaving it in just produces blank/NaN cells in the heatmap
    without telling you why. The corresponding score_* column for that model
    (the continuous score vector) is NOT excluded, since it is not
    necessarily constant even when the thresholded delta_vector is.
    """
    import re

    # Exclude baseline models containing 'tf-' or 'tf_'
    score_cols = [c for c in combined.columns if c.startswith('score_') and not re.search(r'(^|_)tf[-_]', c.lower())]
    delta_cols = [c for c in combined.columns if c.startswith('delta_vector_') and not re.search(r'(^|_)tf[-_]', c.lower())]

    constant_delta_cols = [c for c in delta_cols if combined[c].nunique() <= 1]
    used_delta_cols = [c for c in delta_cols if c not in constant_delta_cols]

    if constant_delta_cols:
        print('NOTE: excluding constant delta_vector column(s) from the delta correlation '
              'heatmaps (model predicted the same class for every image, so correlation '
              'with other models is undefined):')
        for c in constant_delta_cols:
            value = int(combined[c].iloc[0])
            pct = 100 if value == 1 else 0
            print(f'  - {c}: always {value} ({pct}% positive rate) -- excluded')

    return {
        'score_pearson': combined[score_cols].corr(method='pearson'),
        'delta_pearson': combined[used_delta_cols].corr(method='pearson'),
        'delta_spearman': combined[used_delta_cols].corr(method='spearman'),
    }


In [ ]:
def plot_corr_heatmap(corr, title, filename):
    """Render a correlation matrix as a heatmap with the numeric value in each cell."""
    n = len(corr)
    fig, ax = plt.subplots(figsize=(1.3 * n + 2, 1.1 * n + 2))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')

    ax.set_xticks(range(n))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(n))
    ax.set_yticklabels(corr.index)

    for i in range(n):
        for j in range(n):
            value = corr.iloc[i, j]
            text_color = 'white' if abs(value) > 0.6 else 'black'
            ax.text(j, i, f'{value:.2f}', ha='center', va='center', color=text_color)

    fig.colorbar(im, ax=ax, label='correlation')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.show()

def plot_accuracy_bar(combined, dataset_name, title, filename, benchmark_line=None):
    """Bar chart of each model's accuracy.
    For FFHQ (label 0), accuracy is 1 - positive_rate.
    For others (label 1), accuracy is positive_rate.
    """
    delta_cols = [c for c in combined.columns if c.startswith('delta_vector_')]
    rates = combined[delta_cols].mean()

    if 'ffhq' in dataset_name.lower():
        accuracies = 1.0 - rates
        ylabel_text = 'Accuracy (1 - positive rate)'
    else:
        accuracies = rates
        ylabel_text = 'Accuracy (positive rate)'

    accuracies = accuracies.sort_values(ascending=False)
    labels = [c.replace('delta_vector_', '') for c in accuracies.index]

    fig, ax = plt.subplots(figsize=(0.6 * len(accuracies) + 2, 5))
    colors = ['#c44e52' if (r == 0 or r == 1) else '#4c72b0' for r in accuracies.values]
    bars = ax.bar(labels, accuracies.values, color=colors)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel(ylabel_text)
    ax.set_title(title)

    # Fix matplotlib warning by explicitly setting xticks first
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')

    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(1, color='black', linewidth=0.8, linestyle='--', alpha=0.3)

    if benchmark_line is not None:
        ax.axhline(benchmark_line, color='red', linestyle=':', linewidth=1.5, alpha=0.8, label=f'Benchmark ({benchmark_line})')
        ax.legend(loc='lower left')

    for bar, rate in zip(bars, accuracies.values):
        ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.02, f'{rate:.2f}',
                 ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.show()

## Point this at your Drive folder
Edit `MAIN_RESULTS_DIR` below to the folder containing your prediction CSVs. This script goes through your folder, finds the files with names with the pattern "{generator}\_faces\_{model name}_{training family}_predictions.csv"


Example: flux_faces_tf-r50nd_GANDM_predictions.csv

For each generator, it shows the intersection (in terms of filenames) of the csvs. Thus, if the intersection size is not the same as the number of files in the csv, there's usually a missing name or a naming mismatch.

In [ ]:
# MAIN_RESULTS_DIR = '/content/drive/MyDrive/{YOUR RESULTS DRIVE FOLDER NAME}' MAKE SURE YOU INPUT THE NAME HERE
MAIN_RESULTS_DIR = # Your folder here
result_subfolders = sorted([d for d in glob.glob(os.path.join(MAIN_RESULTS_DIR, '*')) if os.path.isdir(d) and d.endswith('_results')])

print(f"Checking filepath intersections across {len(result_subfolders)} folders...\n")

for drive_csv_dir in result_subfolders:
    folder_name = os.path.basename(drive_csv_dir)
    all_csvs = sorted(glob.glob(os.path.join(drive_csv_dir, '*.csv')))

    # Group CSVs by prefix (same logic as main script)
    grouped_csvs = {}
    for path in all_csvs:
        fname = os.path.basename(path)
        if 'patchforensics' in fname.lower():
            continue

        match = re.match(r'([^_]+)_.*', fname)
        group_name = match.group(1).lower() if match else 'ungrouped'

        if group_name not in grouped_csvs:
            grouped_csvs[group_name] = []
        grouped_csvs[group_name].append(path)

    print("=" * 70)
    print(f"Folder: {folder_name}")
    print("=" * 70)

    for group_name, csv_list in grouped_csvs.items():
        print(f"\n--- Group: {group_name} ({len(csv_list)} files) ---")
        if len(csv_list) == 0:
            continue

        common_index = None

        for p in csv_list:
            fname = os.path.basename(p)

            try:
                df = pd.read_csv(p)
            except Exception as e:
                print(f"  [ERROR] Could not read {fname}: {e}")
                continue

            if 'filepath' not in df.columns:
                print(f"  [WARNING] Skipping {fname}: no 'filepath' column.")
                continue

            current_filepaths = set(df['filepath'].dropna().values)

            if common_index is None:
                common_index = current_filepaths
                print(f"  Loaded {fname: <50} -> {len(current_filepaths)} rows. (Initial intersection)")
            else:
                previous_intersection = common_index.copy()
                common_index = common_index.intersection(current_filepaths)
                print(f"  Loaded {fname: <50} -> Intersection size: {len(common_index)}")

                if len(common_index) == 0:
                    print("\n  🚨 ZERO INTERSECTION REACHED! 🚨")
                    print(f"  The file {fname} has NO overlapping filepaths with the previous files.\n")

                    print("  Sample filepaths from previous intersection:")
                    print(f"    {list(previous_intersection)[:3]}")

                    print(f"\n  Sample filepaths from current file ({fname}):")
                    print(f"    {list(current_filepaths)[:3]}")
                    break # Stop checking this group once it hits 0

        if common_index is not None and len(common_index) > 0:
            print(f"  ✅ Success: Final intersection size for {group_name} is {len(common_index)}")


The cell below will generate the Pearson and Spearman correlation plots and save them in your Google Drive. In addition, it will generate bar graphs comparing the model accuracies at a threshold of 0.5

In [ ]:
PLOTS_OUTPUT_DIR = os.path.join(MAIN_RESULTS_DIR, 'plots')
os.makedirs(PLOTS_OUTPUT_DIR, exist_ok=True)

# Find all subfolders ending in '_results'
result_subfolders = [d for d in glob.glob(os.path.join(MAIN_RESULTS_DIR, '*')) if os.path.isdir(d) and d.endswith('_results')]

if not result_subfolders:
    print(f"No subfolders ending in '_results' found in {MAIN_RESULTS_DIR}")
else:
    print(f"Found {len(result_subfolders)} result subfolders: {[os.path.basename(f) for f in result_subfolders]}")

all_individual_accuracies = []

for drive_csv_dir in result_subfolders:
    print(f"\nProcessing folder: {drive_csv_dir}")
    folder_name = os.path.basename(drive_csv_dir)

    all_csvs = sorted(glob.glob(os.path.join(drive_csv_dir, '*.csv')))

    grouped_csvs = {}
    for path in all_csvs:
        fname = os.path.basename(path)
        match = re.match(r'([^_]+)_.*', fname)
        if match:
            group_name = match.group(1).lower()
            if group_name not in grouped_csvs:
                grouped_csvs[group_name] = []
            grouped_csvs[group_name].append(path)
        else:
            if 'ungrouped' not in grouped_csvs:
                grouped_csvs['ungrouped'] = []
            grouped_csvs['ungrouped'].append(path)

    print(f'Found {len(all_csvs)} CSV(s) total in {drive_csv_dir}')

    # --- Build combined tables ---
    combined_dataframes = {}
    for group_name_key, csv_list_val in grouped_csvs.items():
        group_prefix_for_extractor = f"{group_name_key}_"
        combined_df = build_combined_table(csv_list_val, group_prefix_for_extractor)
        if combined_df is not None:
            if combined_df.empty:
                print(f'WARNING: {group_name_key}_combined is empty. Skipping.')
            else:
                print(f'{group_name_key}_combined: {combined_df.shape[0]} images x {combined_df.shape[1]} columns')
                combined_dataframes[group_name_key] = combined_df

    # --- Apply corrections/transformations ---
    for group_name_key, combined_df_val in combined_dataframes.items():
        cols_to_drop = [c for c in combined_df_val.columns if 'patchforensics' in c.lower()]
        if cols_to_drop:
            combined_df_val = combined_df_val.drop(columns=cols_to_drop)
            combined_dataframes[group_name_key] = combined_df_val

    # --- Run correlations and plot heatmaps/bar charts ---
    for group_name_key, combined_df_val in combined_dataframes.items():
        if combined_df_val is not None:
            print(f"\nProcessing correlations for {folder_name}/{group_name_key}...")
            corrs = compute_correlations(combined_df_val)

            score_cols = [c for c in combined_df_val.columns if c.startswith('score_') and not re.search(r'(^|_)tf[-_]', c.lower())]
            score_spearman = combined_df_val[score_cols].corr(method='spearman')

            def clean_label(c):
                c = re.sub(r'^(score|delta_vector)_', '', c)
                c = re.sub(r'^faces_', '', c)
                return c

            score_pearson = corrs['score_pearson'].rename(index=clean_label, columns=clean_label)
            score_spearman = score_spearman.rename(index=clean_label, columns=clean_label)

            plot_corr_heatmap(score_pearson, f'{group_name_key} -- score vector Pearson correlation', os.path.join(PLOTS_OUTPUT_DIR, f'{folder_name}_{group_name_key}_score_correlation_pearson.png'))
            plot_corr_heatmap(score_spearman, f'{group_name_key} -- score vector Spearman correlation', os.path.join(PLOTS_OUTPUT_DIR, f'{folder_name}_{group_name_key}_score_correlation_spearman.png'))

            df_for_bar = combined_df_val.copy()
            df_for_bar.columns = [re.sub(r'^(score|delta_vector)_faces_', r'\1_', c) for c in df_for_bar.columns]
            plot_accuracy_bar(df_for_bar, group_name_key, f'{group_name_key} -- Model Accuracy', os.path.join(PLOTS_OUTPUT_DIR, f'{folder_name}_{group_name_key}_accuracy_rate.png'))

            # --- Save Individual Accuracies ---
            delta_cols = [c for c in combined_df_val.columns if 'delta_vector' in c.lower()]
            rates = combined_df_val[delta_cols].mean()
            accuracies = (1.0 - rates) if 'ffhq' in folder_name.lower() else rates
            for col, acc in accuracies.items():
                detector_clean = re.sub(r'^(delta_vector_)+(faces_)?', '', col, flags=re.IGNORECASE)
                all_individual_accuracies.append({
                    'Subset': f"{folder_name}/{group_name_key}",
                    'Detector': detector_clean,
                    'Accuracy': acc
                })

# --- Display and Save Global Accuracy Table ---
if all_individual_accuracies:
    df_acc_all = pd.DataFrame(all_individual_accuracies)
    global_accuracy_table = df_acc_all.pivot(index='Detector', columns='Subset', values='Accuracy')
    print("\n" + "="*40 + "\nGLOBAL DETECTOR ACCURACY TABLE\n" + "="*40)
    display(global_accuracy_table.style.format('{:.4f}', na_rep='-'))
    global_csv_path = os.path.join(PLOTS_OUTPUT_DIR, 'global_accuracy_table.csv')
    global_accuracy_table.to_csv(global_csv_path)
    print(f"\nSaved global accuracy table to: {global_csv_path}\n")

In [ ]:
# ============================================================
# AUROC pipeline (real-vs-fake, per model, per generator group)
# ============================================================
# Reads each model's score column directly from its own CSV (NOT via the
# inner-joined `combined_dataframes` tables used above for the correlation
# heatmaps). This matters: the inner join restricts every model in a group
# to the intersection of images all models share, which can shrink a model's
# usable image count below the full set. For AUROC we want each model's
# AUROC computed against its OWN full image set -- all ~20K FFHQ real
# images and all ~10K images of the fake subset in question -- not an
# intersection with other models.
#
# Model names are matched CASE-INSENSITIVELY (e.g. "cnndetection_GAN" and
# "CNNDetection_GAN" are treated as the SAME model), since the same model
# shows up with inconsistent capitalization across different CSV filenames.
# The first-seen original casing is kept as the display name for tables/plots.

import numpy as np
from sklearn.metrics import roc_auc_score
import os
import glob
import re
import pandas as pd

raw_scores_by_key = {}     # {group_key: {model_key_lower: pd.Series(score, index=filepath)}}
model_display_names = {}   # {model_key_lower: original-cased name, first one seen}

for drive_csv_dir in result_subfolders:
    folder_name = os.path.basename(drive_csv_dir)
    all_csvs = sorted(glob.glob(os.path.join(drive_csv_dir, '*.csv')))

    grouped_csvs = {}
    for path in all_csvs:
        fname = os.path.basename(path)
        match = re.match(r'([^_]+)_.*', fname)
        group_name = match.group(1).lower() if match else 'ungrouped'

        # Ignore generic stylegan, stylegan2, and stylegan3 groups
        if group_name in ['stylegan', 'stylegan2', 'stylegan3']:
            continue

        grouped_csvs.setdefault(group_name, []).append(path)

    for group_name_key, csv_list_val in grouped_csvs.items():
        group_prefix = f"{group_name_key}_"
        per_model_scores = {}
        seen_models = {}  # {model_key_lower: source filename}
        for path in csv_list_val:
            fname = os.path.basename(path)
            if 'patchforensics' in fname.lower():
                continue
            try:
                df = pd.read_csv(path)
            except Exception as e:
                print(f"ERROR reading {fname}: {e}. Skipping.")
                continue
            if 'filepath' not in df.columns or 'score' not in df.columns:
                print(f"WARNING: {fname} missing filepath/score column. Skipping.")
                continue
            model = extract_model_name(path, group_prefix)
            model_key = model.lower()
            if model_key in seen_models:
                print(f'WARNING: duplicate model "{model}" (case-insensitive match with '
                      f'"{model_display_names.get(model_key, model_key)}") in '
                      f'{folder_name}/{group_name_key} -- skipping {fname} '
                      f'(already used by {seen_models[model_key]})')
                continue
            seen_models[model_key] = fname
            model_display_names.setdefault(model_key, model)
            per_model_scores[model_key] = df.set_index('filepath')['score'].sort_index()
        raw_scores_by_key[f'{folder_name}/{group_name_key}'] = per_model_scores

print(f"Rebuilt raw (non-joined) per-model scores for {len(raw_scores_by_key)} groups "
      "(model names matched case-insensitively):")
for k, v in raw_scores_by_key.items():
    sizes = {model_display_names[mk]: len(s) for mk, s in v.items()}
    print(f"  {k}: {len(v)} models -- image counts: {sizes}")

# Identify the real-image (FFHQ) group -- everything else is a fake-image group.
ffhq_keys = [k for k in raw_scores_by_key if 'ffhq' in k.lower()]
if len(ffhq_keys) != 1:
    raise ValueError(f"Expected exactly one FFHQ (real-image) group, found: {ffhq_keys}")
ffhq_key = ffhq_keys[0]
ffhq_scores = raw_scores_by_key[ffhq_key]
print(f"\nUsing '{ffhq_key}' as the real-image (label=0) reference for AUROC --")
print("each model's full FFHQ score vector, not intersected with other models.")

In [ ]:
# Define our target groups (subsets) and target model families
target_groups = {
    'ffhq_results/ffhq': 'FFHQ',
    'sd2_results/sd2': 'SD2',
    'stylegan23_results/stylegan2-1.0': 'StyleGAN2-1.0',
    'flux_results/flux': 'FLUX'
}

gan_models = ['aide_gan', 'cnndetection_gan', 'co-spy_gan', 'dmimagedetection_gan', 'freqnet_gan', 'grag2021_gan', 'rine4_gan']
dm_models = ['aide_dm', 'co-spy_dm', 'dmimagedetection_dm', 'rine_dm']

# Map same architecture pairs across GAN and DM families
same_arch_pairs = [
    ('aide_gan', 'aide_dm'),
    ('co-spy_gan', 'co-spy_dm'),
    ('dmimagedetection_gan', 'dmimagedetection_dm'),
    ('rine4_gan', 'rine_dm')
]

# Ensure raw_scores_by_key exists, otherwise print a helpful error
if 'raw_scores_by_key' not in globals():
    print("Error: 'raw_scores_by_key' is not defined. Please run cell 'nIwB08pjIlaj' first to populate it.")
else:
    for group_key, group_label in target_groups.items():
        if group_key not in raw_scores_by_key:
            print(f"Group {group_key} not found in raw_scores_by_key. Skipping.")
            continue

        per_model_scores = raw_scores_by_key[group_key]

        # Build a combined aligned dataframe of raw scores for this group using an inner join
        aligned_series = {}
        for model_key, scores_series in per_model_scores.items():
            clean_name = re.sub(r'^faces_', '', model_key).lower()
            if clean_name in gan_models or clean_name in dm_models:
                aligned_series[clean_name] = scores_series

        if not aligned_series:
            print(f"No matching models found for {group_label}.")
            continue

        df_group = pd.DataFrame(aligned_series).dropna()
        print(f"\n=== Spearman Correlation Analysis for {group_label} ({len(df_group)} aligned images) ===")

        # Compute full Spearman correlation matrix
        corr_matrix = df_group.corr(method='spearman')

        # Filter models to those actually present in this dataset's loaded data
        present_gan = [m for m in gan_models if m in corr_matrix.columns]
        present_dm = [m for m in dm_models if m in corr_matrix.columns]

        # 1. GAN-trained within-family correlation
        if len(present_gan) > 1:
            gan_corr = corr_matrix.loc[present_gan, present_gan]
            # Extract upper triangle values excluding the diagonal
            gan_vals = gan_corr.values[np.triu_indices_from(gan_corr, k=1)]
            mean_gan = np.mean(gan_vals)
            print(f"Mean GAN-to-GAN Spearman Correlation ({len(present_gan)} models): {mean_gan:.4f}")
        else:
            print("Not enough GAN models present to compute within-family correlation.")

        # 2. DM-trained within-family correlation
        if len(present_dm) > 1:
            dm_corr = corr_matrix.loc[present_dm, present_dm]
            dm_vals = dm_corr.values[np.triu_indices_from(dm_corr, k=1)]
            mean_dm = np.mean(dm_vals)
            print(f"Mean DM-to-DM Spearman Correlation ({len(present_dm)} models): {mean_dm:.4f}")
        else:
            print("Not enough DM models present to compute within-family correlation.")

        # 2b. DM-trained within-family correlation (excluding dmimagedetection)
        present_dm_no_dmimage = [m for m in present_dm if m != 'dmimagedetection_dm']
        if len(present_dm_no_dmimage) > 1:
            dm_no_dmimage_corr = corr_matrix.loc[present_dm_no_dmimage, present_dm_no_dmimage]
            dm_no_dmimage_vals = dm_no_dmimage_corr.values[np.triu_indices_from(dm_no_dmimage_corr, k=1)]
            mean_dm_no_dmimage = np.mean(dm_no_dmimage_vals)
            print(f"Mean DM-to-DM Spearman Correlation (excluding dmimagedetection) ({len(present_dm_no_dmimage)} models): {mean_dm_no_dmimage:.4f}")
        else:
            print("Not enough DM models (excluding dmimagedetection) present to compute correlation.")

        # 3. Cross-family correlation (GAN-trained vs DM-trained) excluding same-architecture pairs
        if present_gan and present_dm:
            cross_corr = corr_matrix.loc[present_gan, present_dm].copy()

            # Mask out same architecture pairs
            for g_model in present_gan:
                base_g = g_model.replace('_gan', '')
                for d_model in present_dm:
                    base_d = d_model.replace('_dm', '')
                    # Custom check for rine4_gan and rine_dm as same architecture
                    if base_g == base_d or (g_model == 'rine4_gan' and d_model == 'rine_dm'):
                        cross_corr.loc[g_model, d_model] = np.nan

            # Compute mean ignoring the masked NaN pairs
            mean_cross = cross_corr.mean().mean()
            print(f"Mean GAN-to-DM (Cross-Family) Spearman Correlation (excluding same-architecture pairs): {mean_cross:.4f}")
        else:
            print("Missing either GAN or DM models to compute cross-family correlation.")

        # 4. Correlation of the same model architecture trained across different families
        pair_corrs = []
        for g_model, d_model in same_arch_pairs:
            if g_model in corr_matrix.columns and d_model in corr_matrix.columns:
                val = corr_matrix.loc[g_model, d_model]
                pair_corrs.append(val)
                print(f"  - Same model pair {g_model} vs {d_model}: {val:.4f}")
        if pair_corrs:
            print(f"Mean Same-Model Cross-Family Spearman Correlation ({len(pair_corrs)} pairs): {np.mean(pair_corrs):.4f}")
        else:
            print("No matching cross-family same-architecture model pairs present.")

In [ ]:
# Categorize subsets as requested (now including FFHQ)
subset_categories = {
    'Real (FFHQ)': [
        'ffhq_results/ffhq'
    ],
    'GAN': [
        'stylegan1_results/stylegan-0.5',
        'stylegan1_results/stylegan-0.7',
        'stylegan23_results/stylegan2-0.5',
        'stylegan23_results/stylegan2-1.0',
        'stylegan23_results/stylegan3-0.5',
        'stylegan23_results/stylegan3-0.7'
    ],
    'DM': [
        'sd1.5_results/sd1.5',
        'sd2_results/sd2',
        'sdxl_results/sdxl'
    ],
    'Rectified Flow': [
        'flux_results/flux',
        'sd3_results/sd3'
    ]
}

same_arch_pairs = [
    ('aide_gan', 'aide_dm'),
    ('co-spy_gan', 'co-spy_dm'),
    ('dmimagedetection_gan', 'dmimagedetection_dm'),
    ('rine4_gan', 'rine_dm')
]

gan_models = ['aide_gan', 'cnndetection_gan', 'co-spy_gan', 'dmimagedetection_gan', 'freqnet_gan', 'grag2021_gan', 'rine4_gan']
dm_models = ['aide_dm', 'co-spy_dm', 'dmimagedetection_dm', 'rine_dm']

all_results = []
sd_no_dmimage_results = []

for category, groups in subset_categories.items():
    for group_key in groups:
        if group_key not in raw_scores_by_key:
            continue

        per_model_scores = raw_scores_by_key[group_key]
        aligned_series = {}
        for model_key, scores_series in per_model_scores.items():
            clean_name = re.sub(r'^faces_', '', model_key).lower()
            if clean_name in gan_models or clean_name in dm_models:
                aligned_series[clean_name] = scores_series

        if not aligned_series:
            continue

        df_group = pd.DataFrame(aligned_series).dropna()
        corr_matrix = df_group.corr(method='spearman')
        subset_name = group_key.split('/')[-1]

        # 1. Within-Family GAN Mean
        present_gan = [m for m in gan_models if m in corr_matrix.columns]
        mean_gan_within = np.nan
        if len(present_gan) > 1:
            gan_corr = corr_matrix.loc[present_gan, present_gan]
            mean_gan_within = np.mean(gan_corr.values[np.triu_indices_from(gan_corr, k=1)])

        # 2. Within-Family DM Mean
        present_dm = [m for m in dm_models if m in corr_matrix.columns]
        mean_dm_within = np.nan
        if len(present_dm) > 1:
            dm_corr = corr_matrix.loc[present_dm, present_dm]
            mean_dm_within = np.mean(dm_corr.values[np.triu_indices_from(dm_corr, k=1)])

        # 3. Cross-Family (Excluding Same Architecture)
        mean_cross_ex_same = np.nan
        if present_gan and present_dm:
            cross_corr = corr_matrix.loc[present_gan, present_dm].copy()
            for g_model in present_gan:
                base_g = g_model.replace('_gan', '')
                for d_model in present_dm:
                    base_d = d_model.replace('_dm', '')
                    if base_g == base_d or (g_model == 'rine4_gan' and d_model == 'rine_dm'):
                        cross_corr.loc[g_model, d_model] = np.nan
            mean_cross_ex_same = cross_corr.mean().mean()

        # 4. Same-Model Cross-Family Mean
        pair_corrs = []
        for g_model, d_model in same_arch_pairs:
            if g_model in corr_matrix.columns and d_model in corr_matrix.columns:
                pair_corrs.append(corr_matrix.loc[g_model, d_model])
        mean_same_model = np.mean(pair_corrs) if pair_corrs else np.nan

        all_results.append({
            'Category': category,
            'Subset': subset_name,
            'Within GAN Mean': mean_gan_within,
            'Within DM Mean': mean_dm_within,
            'Cross-Family (Excl. Same)': mean_cross_ex_same,
            'Same-Model Cross-Family': mean_same_model
        })

        # Additional calculation for DM (Stable Diffusion) category excluding 'dmimagedetection_dm'
        if category == 'DM':
            present_dm_no_dmimage = [m for m in present_dm if m != 'dmimagedetection_dm']

            # Within-Family DM Mean excluding dmimagedetection_dm
            mean_dm_within_no_dmimage = np.nan
            if len(present_dm_no_dmimage) > 1:
                dm_corr_no_dmimage = corr_matrix.loc[present_dm_no_dmimage, present_dm_no_dmimage]
                mean_dm_within_no_dmimage = np.mean(dm_corr_no_dmimage.values[np.triu_indices_from(dm_corr_no_dmimage, k=1)])

            # Cross-Family excluding same architecture and excluding dmimagedetection_dm
            mean_cross_ex_same_no_dmimage = np.nan
            if present_gan and present_dm_no_dmimage:
                cross_corr_no_dmimage = corr_matrix.loc[present_gan, present_dm_no_dmimage].copy()
                for g_model in present_gan:
                    base_g = g_model.replace('_gan', '')
                    for d_model in present_dm_no_dmimage:
                        base_d = d_model.replace('_dm', '')
                        if base_g == base_d or (g_model == 'rine4_gan' and d_model == 'rine_dm'):
                            cross_corr_no_dmimage.loc[g_model, d_model] = np.nan
                mean_cross_ex_same_no_dmimage = cross_corr_no_dmimage.mean().mean()

            # Same-Model Cross-Family Mean excluding dmimagedetection_dm pairs
            pair_corrs_no_dmimage = []
            for g_model, d_model in same_arch_pairs:
                if g_model in corr_matrix.columns and d_model in present_dm_no_dmimage:
                    pair_corrs_no_dmimage.append(corr_matrix.loc[g_model, d_model])
            mean_same_model_no_dmimage = np.mean(pair_corrs_no_dmimage) if pair_corrs_no_dmimage else np.nan

            sd_no_dmimage_results.append({
                'Subset': subset_name,
                'Within DM Mean (no DMImage)': mean_dm_within_no_dmimage,
                'Cross-Family (no DMImage, Excl. Same)': mean_cross_ex_same_no_dmimage,
                'Same-Model (no DMImage)': mean_same_model_no_dmimage
            })

# Create and format the structured summary table
df_summary = pd.DataFrame(all_results)
display(df_summary.style.format({
    'Within GAN Mean': '{:.4f}',
    'Within DM Mean': '{:.4f}',
    'Cross-Family (Excl. Same)': '{:.4f}',
    'Same-Model Cross-Family': '{:.4f}'
}, na_rep='-'))

if sd_no_dmimage_results:
    print("\n" + "="*80)
    print("SPEARMAN CORRELATION METRICS FOR SD SUBSETS (EXCLUDING DMImageDetection_DM)")
    print("="*80)
    df_sd_no_dmimage = pd.DataFrame(sd_no_dmimage_results)
    display(df_sd_no_dmimage.style.format({
        'Within DM Mean (no DMImage)': '{:.4f}',
        'Cross-Family (no DMImage, Excl. Same)': '{:.4f}',
        'Same-Model (no DMImage)': '{:.4f}'
    }, na_rep='-'))

In [ ]:
# ------------------------------------------------------------
# Baseline models: models whose name contains a "tf-"/"tf_" token (the
# TrueFake benchmark baselines -- tf-clipd, tf-mislnet, tf-npr, tf-r50nd,
# tf-r50e2p). NOTE: because of how the group prefix is stripped, these come
# through as e.g. "faces_tf-clipd_GANDM" rather than literally starting with
# "tf" -- this checks for a "tf-"/"tf_" token anywhere in the name so the
# actual TrueFake baselines are still caught correctly. The check lowercases
# internally, so it's case-insensitive regardless of input casing.
def is_baseline_model(model_name):
    return re.search(r'(^|_)tf[-_]', model_name.lower()) is not None

# ------------------------------------------------------------
# AUROC per model per fake-image group: pairs the group's full fake (label=1)
# score vector with the model's full FFHQ real (label=0) score vector --
# i.e. all available FFHQ images (~20K) vs. all available images of the
# fake subset in question (~10K), each read straight from that model's own
# CSV (see raw_scores_by_key above), not reduced by any cross-model join.
#
# Models are matched via the case-insensitive key already built into
# raw_scores_by_key/model_display_names above, and results are reported
# under the display name (first-seen original casing) -- so e.g.
# "cnndetection_GAN" and "CNNDetection_GAN" are counted and analyzed as ONE
# model, not two.
#
# KNOWN DATA GAP (as of this run): none of the 'tf-' baseline models have any
# predictions on FFHQ (real images) anywhere in Drive, and they only have
# fake-image predictions for the sd2_faces / sdxl_faces groups. So baseline
# AUROC cannot currently be computed for ANY group -- this is expected and
# will resolve itself once baseline-vs-FFHQ predictions are added.
auroc_results = {}          # {group_key: {display_name: auroc}}
auroc_n_counts = {}         # {group_key: {display_name: (n_real, n_fake)}}
missing_ffhq_pairing = {}   # {group_key: [display names skipped]}

for key, per_model_scores in raw_scores_by_key.items():
    if key == ffhq_key:
        continue
    per_model_auroc = {}
    per_model_n = {}
    skipped = []
    for model_key, fake_series in per_model_scores.items():
        display_name = model_display_names[model_key]
        clean_name = re.sub(r'^faces_', '', display_name)

        # Prepend 'faces_' to model_key if not already present for FFHQ lookup
        ffhq_lookup_key = model_key
        if 'faces_' not in ffhq_lookup_key:
            ffhq_lookup_key = 'faces_' + ffhq_lookup_key

        if ffhq_lookup_key not in ffhq_scores:
            skipped.append(clean_name)
            continue
        real_scores = ffhq_scores[ffhq_lookup_key].values
        fake_vals = fake_series.values
        y_true = np.concatenate([np.zeros(len(real_scores)), np.ones(len(fake_vals))])
        y_score = np.concatenate([real_scores, fake_vals])
        try:
            per_model_auroc[clean_name] = roc_auc_score(y_true, y_score)
            per_model_n[clean_name] = (len(real_scores), len(fake_vals))
        except ValueError as e:
            print(f"  Could not compute AUROC for {key}/{clean_name}: {e}")
    auroc_results[key] = per_model_auroc
    auroc_n_counts[key] = per_model_n
    if skipped:
        missing_ffhq_pairing[key] = skipped

print(f"Computed AUROC for {len(auroc_results)} generator groups.\n")
print("Image counts used per model (n_real FFHQ images, n_fake images) -- confirms")
print("the FULL available set is used for each model, not an intersection:")
for key, n_map in auroc_n_counts.items():
    for model, (n_real, n_fake) in n_map.items():
        print(f"  {key}/{model}: n_real={n_real}, n_fake={n_fake}")

if missing_ffhq_pairing:
    print("\nNOTE: the following models have no matching FFHQ (real-image) predictions,")
    print("so their AUROC could not be computed for these groups (this currently includes")
    print("ALL 'tf-' baselines -- see the note in this cell for why):")
    for key, models in missing_ffhq_pairing.items():
        print(f"  {key}: {models}")

# ------------------------------------------------------------
# Table with every AUROC number: rows = models, columns = generator groups.
auroc_table = pd.DataFrame(auroc_results).sort_index()
auroc_table.index.name = 'model'

def highlight_max(s):
    '''
    Highlights the maximum value in a Series in bold.
    '''
    # Ensure that NaN values are not considered for highlighting
    s_numeric = s.dropna()
    if s_numeric.empty:
        return ['' for _ in s] # Return empty styles if no numeric values

    is_max = s == s_numeric.max()
    return ['font-weight: bold' if v else '' for v in is_max]

display(auroc_table.style.format('{:.4f}', na_rep='-').apply(highlight_max, axis=0))

auroc_table.round(4).to_csv(os.path.join(PLOTS_OUTPUT_DIR, 'auroc_table.csv'))
print(f"\nSaved full AUROC table to {os.path.join(PLOTS_OUTPUT_DIR, 'auroc_table.csv')}")


In [ ]:
# ------------------------------------------------------------
# Plot: one bar per non-baseline model's AUROC; the single highest AUROC
# among baseline ("tf-") models is drawn as a red dotted horizontal line
# instead of a bar -- skipped, with a printed note, if no baseline AUROC
# exists yet for that group (currently true for every group -- see the
# note in the previous cell).
def plot_auroc_bar(group_key, per_model_auroc, title, filename):
    baseline_scores = {m: a for m, a in per_model_auroc.items() if is_baseline_model(m)}
    nonbaseline_scores = {m: a for m, a in per_model_auroc.items() if not is_baseline_model(m)}

    if not nonbaseline_scores:
        print(f"[{group_key}] No non-baseline models with AUROC -- skipping plot.")
        return

    items = sorted(nonbaseline_scores.items(), key=lambda kv: kv[1], reverse=True)
    labels = [m for m, _ in items]
    values = [a for _, a in items]

    fig, ax = plt.subplots(figsize=(0.6 * len(labels) + 2, 5))
    bars = ax.bar(labels, values, color='#4c72b0')
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('AUROC')
    ax.set_title(title)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axhline(1, color='black', linewidth=0.8, linestyle='--', alpha=0.3)

    if baseline_scores:
        best_name = max(baseline_scores, key=baseline_scores.get)
        best_auroc = baseline_scores[best_name]
        ax.axhline(best_auroc, color='red', linestyle=':', linewidth=1.5, alpha=0.9,
                   label=f'Best baseline: {best_name} ({best_auroc:.2f})')
        ax.legend(loc='lower left')
    else:
        print(f"[{group_key}] No baseline ('tf-') AUROC available yet -- no red line drawn.")

    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02, f'{value:.2f}',
                ha='center', va='bottom', fontsize=9)

    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.show()

for key, per_model_auroc in auroc_results.items():
    dataset_name = key.split('/')[-1]
    plot_auroc_bar(
        key, per_model_auroc,
        f'{dataset_name} -- Model AUROC',
        os.path.join(PLOTS_OUTPUT_DIR, f'{key.replace("/", "_")}_auroc.png')
    )


**This computes the ROC curve and violin plots**

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve, auc
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if 'PLOTS_OUTPUT_DIR' not in globals():
    PLOTS_OUTPUT_DIR = os.path.join(MAIN_RESULTS_DIR, 'plots')
    os.makedirs(PLOTS_OUTPUT_DIR, exist_ok=True)

ROC_DIR = os.path.join(PLOTS_OUTPUT_DIR, 'roc_curves')
VIOLIN_DIR = os.path.join(PLOTS_OUTPUT_DIR, 'violin_plots')
os.makedirs(ROC_DIR, exist_ok=True)
os.makedirs(VIOLIN_DIR, exist_ok=True)

# Group data by model (detector)
detector_data = {} # display_name -> {group_key -> scores_array}

if 'raw_scores_by_key' in globals():
    for group_key, per_model_scores in raw_scores_by_key.items():
        for model_key, scores_series in per_model_scores.items():
            display_name = model_display_names[model_key]
            if display_name not in detector_data:
                detector_data[display_name] = {}
            detector_data[display_name][group_key] = scores_series.values

print(f"Generating ROC and Violin plots for {len(detector_data)} detectors...")

for detector, group_scores in detector_data.items():
    # Find the original model_key to retrieve matching FFHQ scores
    base_key = next((mk for mk, dn in model_display_names.items() if dn == detector), None)
    if not base_key:
        continue

    ffhq_lookup_key = base_key if 'faces_' in base_key else 'faces_' + base_key

    if 'ffhq_scores' not in globals() or ffhq_lookup_key not in ffhq_scores:
        print(f"Skipping plots for {detector} - no FFHQ (real) scores available.")
        continue

    real_scores = ffhq_scores[ffhq_lookup_key].values
    # Use case-insensitive substitution to remove 'faces_'
    clean_detector = re.sub(r'^faces_', '', detector, flags=re.IGNORECASE)

    # ==========================================
    # 1. ROC Curves
    # ==========================================
    plt.figure(figsize=(8, 6))
    plotted_any = False

    for group_key, scores in group_scores.items():
        if group_key == ffhq_key:
            continue

        y_true = np.concatenate([np.zeros(len(real_scores)), np.ones(len(scores))])
        y_score = np.concatenate([real_scores, scores])

        try:
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            dataset_name = group_key.split('/')[-1]
            plt.plot(fpr, tpr, label=f'{dataset_name} (AUC = {roc_auc:.2f})')
            plotted_any = True
        except Exception as e:
            pass

    if plotted_any:
        plt.plot([0, 1], [0, 1], color='black', linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve: {clean_detector}')
        plt.legend(loc="lower right", fontsize=8)
        plt.grid(True, axis='both', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(ROC_DIR, f'{clean_detector}_roc.png'), dpi=150)
        plt.show()
    else:
        plt.close()

    # ==========================================
    # 2. Violin Plots
    # ==========================================
    plot_data = []

    # Add Real (FFHQ) data first
    plot_data.append(pd.DataFrame({'Score': real_scores, 'Dataset': 'FFHQ (Real)'}))

    # Add all Fake datasets
    for group_key, scores in group_scores.items():
        if group_key == ffhq_key:
            continue
        dataset_name = group_key.split('/')[-1]
        plot_data.append(pd.DataFrame({'Score': scores, 'Dataset': dataset_name}))

    if plot_data:
        df_plot = pd.concat(plot_data, ignore_index=True)
        plt.figure(figsize=(10, 6))

        # Create custom palette to color FFHQ differently
        unique_datasets = df_plot['Dataset'].unique()
        custom_palette = {ds: 'skyblue' for ds in unique_datasets}
        custom_palette['FFHQ (Real)'] = 'lightgreen'

        label_order = ["FFHQ (Real)", "stylegan-0.5", "stylegan-0.7", "stylegan2-0.5", "stylegan2-1.0", "stylegan3-0.5", "stylegan3-0.7", "sd1.5", "sd2", "sdxl", "sd3", "flux"]
        sns.violinplot(x='Dataset', y='Score', order=label_order, data=df_plot, inner="quart", density_norm="width", cut=0, hue='Dataset', palette=custom_palette, legend=False)
        plt.axhline(0.5, color='red', linestyle='--', label='Threshold (0.5)')
        plt.xticks(rotation=45, ha='right')
        plt.title(f'Score Distribution: {clean_detector}')
        plt.ylim(-0.05, 1.05)
        plt.legend(loc='upper right')
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(VIOLIN_DIR, f'{clean_detector}_violin.png'), dpi=150)
        plt.show()